# Chapter 7 &mdash; Subset Construction: NFA to DFA

**Concept 8 of the Chapter 7 decomposition:** *Subset Construction: Converting an NFA to a DFA*

DFA states are <i>sets</i> of NFA states; expand each unexpanded set by Eclose&ndash;move&ndash;Eclose until closed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Subset-Construction/Concept-Subset-Construction.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The **subset construction** makes the token set *itself* the DFA state.

* start state: $Eclosure(Q_0)$;
* for each unexpanded set $S$ and symbol $a$: $Eclosure(\delta(S,a))$ is a new state;
* final: any set meeting $F$;
* repeat until no new set appears.

At most $2^{|Q|}$ sets exist, so it terminates &mdash; and that bound is where the
exponential blow-up of Chapter 5 comes from. In practice only the **reachable** sets
appear, which is usually far fewer.

This construction *is* the proof that NFA add no power.

## 2. Definitions

### The NFA to determinize

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''')

### The construction, written out

In [ ]:
def subset(N):
    start = frozenset(Eclosure(N, N["Q0"]))
    Q, todo, Dl = {start}, [start], {}
    while todo:
        Sset = todo.pop()
        for a in sorted(N["Sigma"]):
            t = frozenset(Eclosure(N, {x for q in Sset for x in step_nfa(N, q, a)}))
            Dl[(Sset, a)] = t
            if t not in Q: Q.add(t); todo.append(t)
    F = {s for s in Q if s & N["F"]}
    return Q, Dl, start, F

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;7.&nbsp;The Language of an NFA: $\hat{\delta}$ via $Eclosure$–$\delta$–$Eclosure$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Delta-Hat-Via-Eclosure/Concept-Delta-Hat-Via-Eclosure.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;9.&nbsp;Theorem: $L$ is Regular iff Some NFA Recognizes It](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Regular-Iff-NFA/Concept-Regular-Iff-NFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The reachable subsets, listed.

In [ ]:
Q, Dl, start, F = subset(N)
print("start set :", sorted(start))
for s in sorted(Q, key=lambda x: (len(x), sorted(x))):
    print("  %-22s %s" % (sorted(s), "FINAL" if s in F else ""))
print("\n%d reachable subsets out of 2^%d = %d possible"
      % (len(Q), len(N["Q"]), 2 ** len(N["Q"])))

`nfa2dfa` produces a machine of the same size and the same language.

In [ ]:
D = nfa2dfa(N)
print("nfa2dfa gives %d states; our construction found %d" % (len(D["Q"]), len(Q)))
assert len(D["Q"]) == len(Q)
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_dfa(D, s) == accepts_nfa(N, s) for s in strs)
print("DFA and NFA agree on all %d strings up to length 10" % len(strs))

Only **reachable** subsets appear &mdash; the $2^{|Q|}$ bound is worst case, not typical.

In [ ]:
print("2^|Q| = %d, reachable = %d, minimized = %d"
      % (2 ** len(N["Q"]), len(D["Q"]), len(min_dfa(D)["Q"])))

But the worst case is real: the $N$-th-last family hits it.

In [ ]:
def nth_last_nfa(k):
    lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> S1']
    for j in range(1, k-1): lines.append('S%d : 0 | 1 -> S%d' % (j, j+1))
    if k > 1: lines.append('S%d : 0 | 1 -> F' % (k-1))
    else:     lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> F']
    return md2mc('\n'.join(lines))

for k in range(1, 5):
    A = nth_last_nfa(k)
    print("k=%d : NFA %2d states -> DFA %2d states (min %2d), 2^k = %d"
          % (k, len(A["Q"]), len(nfa2dfa(A)["Q"]), len(min_dfa(nfa2dfa(A))["Q"]), 2**k))
    assert len(min_dfa(nfa2dfa(A))["Q"]) >= 2**k

Names get long; `shrink_dfastates` renames them.

In [ ]:
print("a raw subset-construction state name :", sorted(D["Q"])[0][:60], "...")

## 4. Animation

The determinized machine &mdash; each state is a whole token set.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(N)), FuseEdges=True)

## 5. Exercises


1. Determinize the $\varepsilon$-NFA of Concept 6 by hand. How many subsets are reachable?
2. Why is $\emptyset$ a legitimate subset-construction state? What is it?
3. When does the subset construction produce **fewer** states than the NFA had?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Subset-Construction')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')